In [ ]:
import requests
import csv
import datetime
import time
import pandas as pd

In [ ]:


def time_to_float(time_str):
    minutes, seconds = map(int, time_str.split(":"))
    return minutes + seconds / 60

def get_player_rating(player):
   #Offensive Stats
    xPoints = (player['goals'] + player['assists']) / player['gamesPlayed']
    xPlusMinus = player['pm'] / player['gamesPlayed']
    #xShots = player['shotsOnGoal'] / player['gamesPlayed'] if 'shotsOnGoal' in player else 0

  #Defensive Stats
    xBlocks = player['blockedShots'] / player['gamesPlayed']
    xToi = player['icetime'] / player['gamesPlayed']
    xHits = player['hits'] / player['gamesPlayed']
    xTkwy = player['takeaways'] / player['gamesPlayed']
    xGvwy = player['giveaways'] / player['gamesPlayed']
    # temporary: avg faceoff needs to take into acc num faceoffs total which is in play by play data not boxscore

    # Faceoff Success for forwards
    xFO = player['AccFaceOffPercentage'] / player['gamesPlayed']

    #penalty impact
    xPim = player['pim'] / player['gamesPlayed']

#idea, should we compare these stats to salary? A "moneyball appraoch might give us a better idea of how well a team is going to do"
    # Defensemen
     if player['position'] == 'D':
        offensive_rating = (0.3 * xPoints) + (0.1 * xPlusMinus)
        defensive_rating = (0.25 * xBlocks) + (0.2 * xToi) + (0.15 * xHits) + (0.15 * xTkwy) - (0.4 * xGvwy) - (0.15 * xPim)
 # Forwards
    else:
        offensive_rating = (0.4 * xPoints) + (0.2 * xShots) + (0.1 * xPlusMinus)
        defensive_rating = (0.05 * xBlocks) + (0.15 * xToi) + (0.05 * xHits) + (0.15 * xTkwy) - (0.4 * xGvwy) - (0.15 * xPim) + (0.1 * (xFO - 0.5))

    return offensive_rating + defensive_rating
def get_all_player_rating(game_data, away, year):
    player_ids = []
    for forward in game_data["playerByGameStats"]["awayTeam" if away else "homeTeam"]["forwards"]:
        player_ids.append(forward["playerId"])
    for defenseman in game_data["playerByGameStats"]["awayTeam" if away else "homeTeam"]["defense"]:
        player_ids.append(defenseman["playerId"])
    players = pd.read_csv(f'{year}_{year+1}_All_Players_Data.csv')

    player_rating = 0
    for playerId in player_ids:
        index = players.loc[players['playerId'] == playerId].index
        if not index.empty:
            player_rating += get_player_rating(players.loc[index[0]])
    return player_rating

def get_goalie_rating(game_data, away, year):
     goalies = pd.read_csv(f'{year}_{year+1}_All_Goalies_Data_TEST.csv')
     goalie_count = 0
     rating = 0
     for goalie in game_data["playerByGameStats"]["awayTeam" if away else "homeTeam"]["goalies"]:
        if time_to_float(goalie["toi"]) > 0:
            goalie_count += 1
            index = goalies.loc[goalies['playerId'] == goalie["playerId"]].index
            if not index.empty:
                goalie_row = goalies.loc[index[0]]  # Extract first matching row
                rating += ( (.005 * goalie_row['gamesPlayed']) + (goalie_row['savePercentage']) - (.2 * goalie_row['GAA']) - (.3 * goalie_row['reboundPercentage']) )

     if goalie_count == 0:
        for goalie in game_data["playerByGameStats"]["awayTeam" if away else "homeTeam"]["goalies"]:
            index = goalies.loc[goalies['playerId'] == goalie["playerId"]].index
            if not index.empty:
                goalie_row = goalies.loc[index[0]]  # Extract first matching row
                rating += ( (.005 * goalie_row['gamesPlayed']) + (goalie_row['savePercentage']) - (.2 * goalie_row['GAA']) - (.3 * goalie_row['reboundPercentage']) )
            goalie_count += 1

     return rating / goalie_count

def get_all_game_scores(year, year2):

    games = pd.DataFrame(columns=['Away', 'Away_Score', 'Home', 'Home_Score', 'gameId', 'Away_xPlayerRating', 'Away_GoalieRating', 'Home_xPlayerRating', 'Home_GoalieRating'])

    start_date = datetime.date(year, 10, 1)
    end_date = datetime.date(year2, 4, 15)
    current_date = start_date

    game_file =  current_date.strftime("%Y") + '_' + end_date.strftime("%Y") + '_All_Game_Scores.csv'
    # Get all teams

    base_url = 'https://api-web.nhle.com/v1/score/'

    game_count = 0
    while current_date <= end_date:
        date_as_str = current_date.strftime("%Y-%m-%d")
        url = f"{base_url}{date_as_str}"

        res = requests.get(url)
        data = res.json()



        for game in data.get('games', []):

            if game["gameType"] == 1:
                continue
            game_count += 1

            game_id = game["id"]
            game_url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/boxscore"

            res = requests.get(game_url)
            game_data = res.json()

            if game_count % 100 == 0:
                print(f"Processed {game_count} games so far...")
            away_players_rating = get_all_player_rating(game_data, True, year)
            home_players_rating = get_all_player_rating(game_data, False, year)
            away_goalie_rating = get_goalie_rating(game_data, True, year)
            home_goalie_rating = get_goalie_rating(game_data, False, year)

            new_row = pd.DataFrame([{'Away': game["awayTeam"]["abbrev"], 'Away_Score': game["awayTeam"]["score"],
                                    'Home': game["homeTeam"]["abbrev"], 'Home_Score': game["homeTeam"]["score"],
                                    'gameId': game["id"], 'Away_xPlayerRating': away_players_rating,
                                    'Away_GoalieRating': away_goalie_rating, 'Home_xPlayerRating': home_players_rating,
                                    'Home_GoalieRating': home_goalie_rating}])
            games.loc[len(games)] = new_row.iloc[0]

        current_date += datetime.timedelta(days=1)

    games.to_csv(game_file, index=False)

    print(f"CSV file '{game_file}' created successfully.")

'''
for i in range(2014, 2019):
    get_all_game_scores(i, i+1)
for i in range(2020, 2024):
    get_all_game_scores(i, i+1)
'''